
# IntermediateOrbitsTools — Complete Class & Function Reference

This notebook is a standalone reference for the **IntermediateOrbitsTools** helper package.

It lists:

- every class,
- the attributes and properties available on each class,
- every class method,
- every public module-level function,
- every internal helper function,
- required and optional arguments,
- a short description of what each item does.

Items are ordered approximately from **most useful/common in Intermediate Orbits coursework** to **least commonly called directly**.

## Package conventions

Unless a function explicitly says otherwise:

- distance: **km**
- velocity: **km/s**
- time: **s**
- angular momentum: **km²/s**
- angles: **radians internally**
- `*_deg` properties/arguments use degrees
- Cartesian orbital states are `[rx, ry, rz, vx, vy, vz]`

The LVLH convention used by this package is **RTN**:

- `+R / +x`: radial outward
- `+T / +y`: along-track
- `+N / +z`: orbit normal / angular-momentum direction

The orbital-plane frame is the classical **PQW/perifocal** frame.

> This reference reflects the current helper package together with the latest expanded `ClassicalOrbitalElements` implementation.



# Quick import

Most common helpers are re-exported from the package root:


In [ ]:
from IntermediateOrbitsTools import (
    ClassicalOrbitalElements,
    StateVector,
    PropagationResult,
    propagate_two_body,
    propagate_universal_anomaly,
    universal_anomaly_state,
    propagate_j2,
    coe_to_rv,
    rv_to_coe,
    eci_to_lvlh,
    relative_state_eci_to_lvlh,
    eci_to_lvlh_dcm,
    perifocal_to_eci_dcm,
    TLERecord,
    CesiumScene,
)


Some less-common helpers are imported from their individual modules:

```python
from IntermediateOrbitsTools.OrbitalElements import (
    eccentric_anomaly_from_true,
    true_anomaly_from_eccentric,
    mean_anomaly_from_eccentric,
    eccentric_anomaly_from_mean,
)

from IntermediateOrbitsTools.OrbitalFrames import (
    lvlh_to_eci_dcm,
    julian_date,
    gmst_rad,
)

from IntermediateOrbitsTools.Propagators import (
    two_body_rhs,
    j2_rhs,
)

from IntermediateOrbitsTools.SpaceTrackTools import SpaceTrackTools
```



# Class ranking

| Rank | Class | Module | Typical use |
|---:|---|---|---|
| 1 | `ClassicalOrbitalElements` | `OrbitalElements.py` | Store any valid six-element classical orbit description and automatically access equivalent/derived orbital quantities. |
| 2 | `StateVector` | `OrbitalElements.py` | Store Cartesian position and velocity together. |
| 3 | `PropagationResult` | `Propagators.py` | Work with the output of two-body, J2, or custom `solve_ivp` propagation. |
| 4 | `TLERecord` | `TLETools.py` | Parse TLEs and generate SGP4 TEME states. |
| 5 | `CesiumScene` | `CesiumOrbit.py` | Build CZML and CesiumJS orbit visualizations. |
| 6 | `SpaceTrackTools` | `SpaceTrackTools.py` | Use the existing PolySpace Space-Track pipeline from normal Python code. |



# 1. `ClassicalOrbitalElements`

**Module:** `OrbitalElements.py`

The main orbit-description class.

The class accepts **exactly six independent orbital quantities**, converts them to the canonical internal set

\[
(a,\ e,\ i,\ \Omega,\ \omega,\ \nu)
\]

and then automatically exposes other equivalent quantities such as:

- specific angular momentum \(h\),
- semi-latus rectum \(p\),
- periapsis/apoapsis radius,
- eccentric anomaly \(E\),
- mean anomaly \(M\),
- mean motion,
- period,
- current orbital radius,
- specific orbital energy.

## Constructor

```python
ClassicalOrbitalElements(
    *elements,
    names=None,
    angle_unit="rad",
    mu_km3_s2=MU_EARTH_KM3_S2,
    **named_elements,
)
```

### Required input

You must provide **exactly six independent orbital quantities** using one of these forms:

```python
ClassicalOrbitalElements(a, e, i, raan, argp, nu)
```

```python
ClassicalOrbitalElements(
    [a, e, i, raan, argp, nu]
)
```

```python
ClassicalOrbitalElements(
    [h, e, i, raan, argp, nu],
    names=["h", "e", "i", "raan", "argp", "nu"],
)
```

```python
ClassicalOrbitalElements(
    {
        "a_km": 7000,
        "e": 0.01,
        "i_deg": 45,
        "raan_deg": 20,
        "argp_deg": 30,
        "M_deg": 10,
    }
)
```

### Accepted orbit-size descriptions

| Input pair | Meaning |
|---|---|
| `a + e` | semi-major axis and eccentricity |
| `h + e` | specific angular momentum and eccentricity |
| `p + e` | semi-latus rectum and eccentricity |
| `rp + ra` | periapsis and apoapsis radius; `a` and `e` are derived |

### Accepted phase descriptions

Exactly one of:

| Input | Meaning |
|---|---|
| `nu`, `theta`, `true_anomaly` | true anomaly |
| `E`, `eccentric_anomaly` | eccentric anomaly |
| `M`, `mean_anomaly` | mean anomaly |

### Optional constructor arguments

| Argument | Default | Description |
|---|---|---|
| `names` | `None` | Names corresponding to a six-value sequence. Required if the sequence is not the default `[a,e,i,raan,argp,nu]`. |
| `angle_unit` | `"rad"` | Unit for angular inputs whose names do not explicitly include `_deg` or `_rad`. |
| `mu_km3_s2` | Earth μ | Gravitational parameter used for conversions involving `h`, period, energy, etc. |



## `ClassicalOrbitalElements` stored attributes

These are the canonical values stored directly by the object.

| Attribute | Units | Description |
|---|---|---|
| `a_km` | km | Semi-major axis. |
| `e` | dimensionless | Eccentricity. |
| `i_rad` | rad | Inclination. |
| `raan_rad` | rad | Right ascension of the ascending node. |
| `argp_rad` | rad | Argument of periapsis. |
| `nu_rad` | rad | True anomaly. |
| `_mu_km3_s2` | km³/s² | Internally stored gravitational parameter. Normally access through `mu_km3_s2`. |



## `ClassicalOrbitalElements` derived attributes / properties

All of these are calculated automatically from the canonical orbit.

| Attribute | Units | Description |
|---|---|---|
| `mu_km3_s2` | km³/s² | Gravitational parameter associated with the object. |
| `inclination_deg` | deg | Inclination in degrees. |
| `raan_deg` | deg | RAAN in degrees. |
| `argument_of_periapsis_deg` | deg | Argument of periapsis in degrees. |
| `true_anomaly_deg` | deg | True anomaly in degrees. |
| `eccentric_anomaly_rad` | rad | Eccentric anomaly \(E\). |
| `eccentric_anomaly_deg` | deg | Eccentric anomaly \(E\) in degrees. |
| `mean_anomaly_rad` | rad | Mean anomaly \(M\). |
| `mean_anomaly_deg` | deg | Mean anomaly \(M\) in degrees. |
| `p_km` | km | Semi-latus rectum \(p=a(1-e^2)\). |
| `h_km2_s` | km²/s | Specific angular momentum magnitude \(\sqrt{\mu p}\). |
| `periapsis_radius_km` | km | \(r_p=a(1-e)\). |
| `apoapsis_radius_km` | km | \(r_a=a(1+e)\). |
| `radius_km` | km | Current orbital radius at the stored true anomaly. |
| `mean_motion_rad_s` | rad/s | Keplerian mean motion. |
| `mean_motion_rev_day` | rev/day | Keplerian mean motion in revolutions per day. |
| `period_s` | s | Keplerian orbital period. |
| `period_min` | min | Keplerian orbital period in minutes. |
| `specific_orbital_energy_km2_s2` | km²/s² | Specific orbital energy \(-\mu/(2a)\). |
| `degrees` | dict | Canonical COEs with all angular values in degrees. |
| `derived` | dict | Collection of commonly useful derived quantities. |
| `vector6` | ndarray | `[a, e, i, RAAN, argp, nu]` with angles in radians. |



## `ClassicalOrbitalElements` short aliases

These aliases are convenient for textbook notation.

| Alias | Same as | Units |
|---|---|---|
| `a` | `a_km` | km |
| `h` | `h_km2_s` | km²/s |
| `p` | `p_km` | km |
| `rp` | `periapsis_radius_km` | km |
| `ra` | `apoapsis_radius_km` | km |
| `i` | `i_rad` | rad |
| `raan` | `raan_rad` | rad |
| `argp` | `argp_rad` | rad |
| `nu` | `nu_rad` | rad |
| `theta` | `nu_rad` | rad |
| `E` | `eccentric_anomaly_rad` | rad |
| `M` | `mean_anomaly_rad` | rad |
| `eccentric_anomaly` | `eccentric_anomaly_rad` | rad |
| `mean_anomaly` | `mean_anomaly_rad` | rad |



## `ClassicalOrbitalElements` methods

| Method / property | Required arguments | Optional arguments | Description |
|---|---|---|---|
| `from_degrees(...)` | `a_km, e, i_deg, raan_deg, argp_deg, nu_deg` | `mu_km3_s2` | Convenience constructor for the standard COE set in degrees. |
| `to_state(...)` | none | `mu_km3_s2=None` | Convert the orbit to a Cartesian `StateVector`. Uses the object's stored μ unless overridden. |
| `vector6` | none | — | Return canonical six-element NumPy array. |
| `degrees` | none | — | Return canonical COEs as a degree-based dictionary. |
| `derived` | none | — | Return a dictionary containing many derived orbital quantities. |

### Internal class helpers

These support flexible construction and normally should not be called directly.

| Method | Required arguments | Description |
|---|---|---|
| `_sequence_to_mapping(values, names)` | `values, names` | Convert six sequence values into a name/value mapping. |
| `_prepare_input(positional, named_elements, names)` | all three | Normalize positional, iterable, mapping, or keyword constructor forms. |
| `_get_angle(values, name, default_unit)` | all three | Retrieve one angle and convert it to radians. |
| `_canonicalize(raw, angle_unit, mu)` | all three | Convert a valid six-quantity input set to canonical `a,e,i,RAAN,argp,nu`. |


In [ ]:
# Example: homework-style h/e input
A = ClassicalOrbitalElements(
    [51400, 0.0006387, 51.65, 15, 157, 15],
    names=["h", "e", "i", "raan", "argp", "theta"],
    angle_unit="deg",
)

print("a =", A.a_km, "km")
print("h =", A.h, "km^2/s")
print("E =", A.eccentric_anomaly_deg, "deg")
print("M =", A.mean_anomaly_deg, "deg")
print("T =", A.period_min, "min")



# 2. `StateVector`

**Module:** `OrbitalElements.py`

Small container for a Cartesian orbital state.

## Constructor

```python
StateVector(r_km, v_km_s)
```

| Argument | Required? | Description |
|---|---|---|
| `r_km` | yes | 3-element Cartesian position vector in km. |
| `v_km_s` | yes | 3-element Cartesian velocity vector in km/s. |

## Attributes and properties

| Attribute | Description |
|---|---|
| `r_km` | 3-element position vector. |
| `v_km_s` | 3-element velocity vector. |
| `vector6` | Combined `[rx,ry,rz,vx,vy,vz]` NumPy array. |

## Methods

| Method | Required arguments | Description |
|---|---|---|
| `from_vector6(state)` | one six-value state | Build a `StateVector` from a six-element iterable. |
| `__post_init__()` | none | Internal validation ensuring both vectors have exactly three values. |


# 3. `PropagationResult`

**Module:** `Propagators.py`

Returned by `propagate_two_body`, `propagate_universal_anomaly`, `propagate_j2`, and `propagate_custom`.

## Stored attributes

| Attribute | Description |
|---|---|
| `times_s` | 1D NumPy array of propagation times. |
| `states` | `N × 6` array containing `[r,v]` at every output time. |
| `raw_solution` | For SciPy propagation, the original `solve_ivp` result. For universal-anomaly propagation, a metadata dictionary describing the method, time step, μ, and time span. |

## Properties

| Property | Description |
|---|---|
| `positions_km` | `N × 3` view of position history. |
| `velocities_km_s` | `N × 3` view of velocity history. |
| `final_state` | Last propagated sample returned as a `StateVector`. |


# `propagate_two_body()` propagation methods

`propagate_two_body()` now supports both numerical `solve_ivp` integration and a **Universal Anomaly / Universal Variable** two-body propagator.

```python
propagate_two_body(
    initial_state,
    t_span_s,
    *,
    times_s=None,
    mu_km3_s2=MU_EARTH_KM3_S2,
    rtol=1e-10,
    atol=1e-12,
    method="DOP853",
    time_step_s=None,
    **solve_ivp_kwargs,
)
```

## Available `method` choices

| `method` | Type | Main user control | Description |
|---|---|---|---|
| `"DOP853"` | `solve_ivp` | `rtol`, `atol` | Default high-order explicit Runge-Kutta integrator. Good general-purpose choice for smooth orbital motion. |
| `"RK45"` | `solve_ivp` | `rtol`, `atol` | Standard adaptive Runge-Kutta 4(5). |
| `"RK23"` | `solve_ivp` | `rtol`, `atol` | Lower-order adaptive Runge-Kutta method. |
| `"Radau"` | `solve_ivp` | `rtol`, `atol` | Implicit Runge-Kutta method, mainly useful for stiff systems. |
| `"BDF"` | `solve_ivp` | `rtol`, `atol` | Implicit multistep method for stiff systems. |
| `"LSODA"` | `solve_ivp` | `rtol`, `atol` | Automatically switches between nonstiff and stiff integration. |
| `"universal_anomaly"` | Universal variables | `time_step_s` | Two-body Kepler propagation using the universal anomaly, Stumpff functions, and Lagrange `f`/`g` coefficients. Does **not** use `rtol` or `atol`. |

Universal-anomaly aliases are also accepted:

```text
"universal"
"universal_variable"
"universal_variables"
```

## Numerical `solve_ivp` example

```python
result = propagate_two_body(
    initial_state,
    (0.0, 86400.0),
    times_s=np.linspace(0.0, 86400.0, 1000),
    method="DOP853",
    rtol=1e-10,
    atol=1e-12,
)
```

## Universal Anomaly example

```python
result = propagate_two_body(
    initial_state,
    (0.0, 86400.0),
    method="universal_anomaly",
    time_step_s=30.0,
)
```

For the universal-anomaly method:

- `time_step_s` is **required** and must be positive.
- Do not provide `times_s`; the output grid is generated from `time_step_s`.
- `rtol` and `atol` are not used.
- Every output state is propagated directly from the initial state rather than repeatedly stepping from the previous output state.
- The final time is always included, even when the requested time span is not an exact multiple of `time_step_s`.
- Negative/final times are supported for backward propagation.

Because universal-variable propagation is a two-body Kepler solution, it is only available through `propagate_two_body()` and not through `propagate_j2()`.


In [ ]:
# Example: Universal Anomaly propagation for one orbit
#
# coes = ClassicalOrbitalElements(
#     [7000, 0.01, 45, 20, 30, 10],
#     angle_unit="deg",
# )
#
# result = propagate_two_body(
#     coes.to_state(),
#     (0.0, coes.period_s),
#     method="universal_anomaly",
#     time_step_s=10.0,
# )
#
# result.positions_km



# 4. `TLERecord`

**Module:** `TLETools.py`

Represents one two-line element set and provides SGP4 state generation.

Requires the third-party `sgp4` package when SGP4 calculations are requested.

## Constructor

```python
TLERecord(line1, line2, name=None)
```

| Argument | Required? | Description |
|---|---|---|
| `line1` | yes | TLE line beginning with `"1 "`. |
| `line2` | yes | TLE line beginning with `"2 "`. |
| `name` | no | Optional object name. |

## Attributes / properties

| Attribute | Description |
|---|---|
| `line1` | Cleaned TLE line 1. |
| `line2` | Cleaned TLE line 2. |
| `name` | Optional object name. |
| `satrec` | SGP4 `Satrec` object created from the TLE. |
| `norad_id` | NORAD catalog ID extracted through SGP4. |
| `epoch` | TLE epoch as a UTC `datetime`. |

## Methods

| Method | Required arguments | Description |
|---|---|---|
| `from_text(text)` | `text` | Parse either a 2-line TLE or name + 2-line TLE block. |
| `state_at(time)` | `datetime` | Return TEME `StateVector` from SGP4 at the requested time. |
| `state_at_epoch()` | none | SGP4 state at the TLE's own epoch. |
| `coes_at(time)` | `datetime` | Convert SGP4 TEME state at a time to osculating two-body `ClassicalOrbitalElements`. |
| `coes_at_epoch()` | none | Same conversion at the TLE epoch. |
| `propagate(start, stop, step_s)` | all three | Sample SGP4 TEME position and velocity over a uniform datetime grid. |
| `__post_init__()` | none | Internal TLE format cleanup/validation. |



# 5. `CesiumScene`

**Module:** `CesiumOrbit.py`

Collects one or more trajectories and writes CZML or a standalone CesiumJS HTML viewer.

## Constructor

```python
CesiumScene(
    name="Intermediate Orbits",
    multiplier=60.0,
)
```

No arguments are required.

| Attribute | Default | Description |
|---|---:|---|
| `name` | `"Intermediate Orbits"` | CZML document / scene name. |
| `multiplier` | `60.0` | Cesium animation clock multiplier. |
| `_packets` | empty list | Internal trajectory packet collection. |
| `_start` | `None` | Internal earliest scene time. |
| `_stop` | `None` | Internal latest scene time. |

## Methods / properties

| Method / property | Required arguments | Optional arguments | Description |
|---|---|---|---|
| `add_trajectory(...)` | `times, positions_km`, keyword-only `entity_id` | `name`, `reference_frame`, `color`, `path_width`, `point_pixel_size` | Add a time-dependent orbit path to the scene. |
| `czml` | none | — | Build the complete CZML packet list. Requires at least one trajectory. |
| `save_czml(path)` | `path` | — | Write the scene to a `.czml` file. |
| `save_html(path, ...)` | `path` | `cesium_version="latest"`, `title=None` | Write a standalone HTML viewer with embedded CZML orbit data. |



# 6. `SpaceTrackTools`

**Module:** `SpaceTrackTools.py`

Python-facing wrapper around the existing PolySpace Space-Track functions. It intentionally does not duplicate the repository's cache/fetch logic.

Import with:

```python
from IntermediateOrbitsTools.SpaceTrackTools import SpaceTrackTools
```

## Constructor

```python
SpaceTrackTools(
    repo_root=None,
    fetching_dir=None,
    gp_history_cache="data/gp_history",
    satcat_cache="data/satcat",
)
```

No constructor arguments are strictly required.

| Attribute | Description |
|---|---|
| `repo_root` | Resolved root of the PolySpace repository. |
| `fetching_dir` | Folder containing function-enabled `FetchGPHistory.py`, `FetchSATCAT.py`, `BuildRealDataset.py`, and `GPDataManager.py`. |
| `gp_history_cache` | GP_HISTORY cache path. |
| `satcat_cache` | SATCAT cache path. |
| `GPHistoryCache` | Imported PolySpace cache class. |
| `SatCatCache` | Imported PolySpace SATCAT cache class. |
| `_fetch_gp_history` | Internal imported function. |
| `_fetch_satcat` | Internal imported function. |
| `_build_real_dataset` | Internal imported function. |
| `_manage_caches` | Internal imported function. |

## Methods

| Method | Required arguments | Optional arguments | Description |
|---|---|---|---|
| `status()` | none | — | Print/return current GP_HISTORY and SATCAT cache status. |
| `fetch_gp_history(**kwargs)` | depends on underlying fetch call | any underlying `fetch_gp_history` keyword | Fetch required GP_HISTORY blocks while automatically supplying this object's cache path. |
| `fetch_satcat(...)` | none | `update_satcat=False`, `dry_run=False` | Reuse/create SATCAT or explicitly refresh it. |
| `build_real_dataset(...)` | `epoch` | `tolerance_hours`, `sample_mode`, `output_dir`, `output_name`, `object_types`, plus filter kwargs | Build the real reference dataset using local caches. |



# Module-level functions — ranked by likely use

The tables below include every module-level function. Functions beginning with `_` are internal helpers and appear in the final section.


## Tier 1 — very common orbital-workflow functions

| Rank | Function | Module | Required arguments | Optional arguments | Description |
|---:|---|---|---|---|---|
| 1 | `propagate_two_body` | `Propagators` | `initial_state, t_span_s` | `times_s`, `mu_km3_s2`, `rtol`, `atol`, `method`, `time_step_s`, other `solve_ivp` kwargs | Point-mass propagation. Use a SciPy method with `rtol`/`atol`, or `method="universal_anomaly"` with `time_step_s`. |
| 2 | `propagate_universal_anomaly` | `Propagators` | `initial_state, t_span_s`, keyword-only `time_step_s` | `mu_km3_s2` | Direct interface to the fixed-output-spacing Universal Anomaly / Universal Variable two-body propagator. |
| 3 | `propagate_j2` | `Propagators` | `initial_state, t_span_s` | `times_s`, `mu_km3_s2`, `radius_km`, `j2`, tolerances, method, other `solve_ivp` kwargs | Numerically propagate with two-body gravity plus J2. |
| 4 | `coe_to_rv` | `OrbitalElements` | `coes` | `mu_km3_s2=None` | Convert `ClassicalOrbitalElements` to a Cartesian `StateVector`. |
| 5 | `rv_to_coe` | `OrbitalElements` | `r_eci_km, v_eci_km_s` | `mu_km3_s2`, `tolerance` | Convert a Cartesian ECI state to elliptical classical orbital elements. |
| 6 | `relative_state_eci_to_lvlh` | `OrbitalFrames` | `chief_r_eci_km, chief_v_eci_km_s, deputy_r_eci_km, deputy_v_eci_km_s` | none | Convert full relative position and velocity into the rotating chief LVLH/RTN frame, including the frame-rotation term in relative velocity. |
| 7 | `eci_to_lvlh` | `OrbitalFrames` | `vector_eci, r_eci_km, v_eci_km_s` | none | Rotate any ECI vector into the spacecraft LVLH/RTN frame. |
| 8 | `eci_to_lvlh_dcm` | `OrbitalFrames` | `r_eci_km, v_eci_km_s` | none | Build the ECI → LVLH/RTN direction cosine matrix. |
| 9 | `orbital_period_s` | `OrbitalElements` | `a_km` | `mu_km3_s2` | Return the Keplerian orbital period in seconds. |
| 10 | `mean_motion_rad_s` | `OrbitalElements` | `a_km` | `mu_km3_s2` | Return Keplerian mean motion in rad/s. |


## Tier 2 — common frame, propagation, and visualization helpers

| Rank | Function | Module | Required arguments | Optional arguments | Description |
|---:|---|---|---|---|---|
| 11 | `universal_anomaly_state` | `Propagators` | `initial_state, delta_t_s` | `mu_km3_s2` | Propagate one state forward or backward by a specified time using the Universal Anomaly solution. |
| 12 | `lvlh_to_eci` | `OrbitalFrames` | `vector_lvlh, r_eci_km, v_eci_km_s` | none | Rotate an LVLH/RTN vector back into ECI. |
| 13 | `lvlh_to_eci_dcm` | `OrbitalFrames` | `r_eci_km, v_eci_km_s` | none | Return the LVLH/RTN → ECI DCM. |
| 14 | `perifocal_to_eci_dcm` | `OrbitalFrames` | `i_rad, raan_rad, argp_rad` | none | Build the PQW/perifocal → ECI DCM. |
| 15 | `eci_to_perifocal_dcm` | `OrbitalFrames` | `i_rad, raan_rad, argp_rad` | none | Build the ECI → PQW/perifocal DCM. |
| 16 | `eci_to_orbital_plane` | `OrbitalFrames` | `vector_eci, i_rad, raan_rad, argp_rad` | none | Express an ECI vector in the orbital-plane/PQW frame. |
| 17 | `orbital_plane_to_eci` | `OrbitalFrames` | `vector_pqw, i_rad, raan_rad, argp_rad` | none | Express a PQW vector in ECI. |
| 18 | `propagate_custom` | `Propagators` | `initial_state, t_span_s, rhs` | `times_s`, `rtol`, `atol`, `method`, `args`, other `solve_ivp` kwargs | Generic six-state `solve_ivp` wrapper for user-defined equations of motion. |
| 19 | `times_from_seconds` | `CesiumOrbit` | `epoch, times_s` | none | Convert propagation seconds into UTC datetimes for Cesium. |
| 20 | `trajectory_packet` | `CesiumOrbit` | `times, positions_km`, keyword-only `entity_id` | `name`, `reference_frame`, `color`, `path_width`, `point_pixel_size` | Build one time-dynamic CZML orbit packet. |
| 21 | `parse_tle_file` | `TLETools` | `path` | none | Parse a file containing repeated 2-line or named 3-line TLE records. |


## Tier 3 — anomaly conversion and dynamics building blocks

| Rank | Function | Module | Required arguments | Optional arguments | Description |
|---:|---|---|---|---|---|
| 22 | `eccentric_anomaly_from_true` | `OrbitalElements` | `nu_rad, e` | none | Convert true anomaly \(\nu\) to eccentric anomaly \(E\). |
| 23 | `true_anomaly_from_eccentric` | `OrbitalElements` | `E_rad, e` | none | Convert eccentric anomaly \(E\) to true anomaly \(\nu\). |
| 24 | `mean_anomaly_from_eccentric` | `OrbitalElements` | `E_rad, e` | none | Compute \(M=E-e\sin E\). |
| 25 | `eccentric_anomaly_from_mean` | `OrbitalElements` | `M_rad, e` | `tolerance=1e-13`, `max_iterations=50` | Solve Kepler's equation for eccentric anomaly. |
| 26 | `two_body_acceleration` | `Propagators` | `r_km` | `mu_km3_s2` | Point-mass gravitational acceleration. |
| 27 | `j2_acceleration` | `Propagators` | `r_km` | `mu_km3_s2`, `radius_km`, `j2` | J2 perturbing acceleration only. |
| 28 | `two_body_rhs` | `Propagators` | `t_s, state` | `mu_km3_s2` | Six-state ODE derivative for point-mass gravity. |
| 29 | `j2_rhs` | `Propagators` | `t_s, state` | `mu_km3_s2`, `radius_km`, `j2` | Six-state ODE derivative for point-mass gravity + J2. |


## Tier 4 — generic DCM and Earth-rotation helpers

| Rank | Function | Module | Required arguments | Optional arguments | Description |
|---:|---|---|---|---|---|
| 30 | `dcm_from_axes` | `OrbitalFrames` | `x_axis, y_axis, z_axis` | none | Build a DCM from three right-handed orthogonal unit directions. |
| 31 | `unit` | `OrbitalFrames` | `vector` | keyword-only `name="vector"` | Normalize a 3-vector and reject zero magnitude. |
| 32 | `rot1` | `OrbitalFrames` | `angle_rad` | none | Active right-handed rotation about +x. |
| 33 | `rot2` | `OrbitalFrames` | `angle_rad` | none | Active right-handed rotation about +y. |
| 34 | `rot3` | `OrbitalFrames` | `angle_rad` | none | Active right-handed rotation about +z. |
| 35 | `eci_to_ecef` | `OrbitalFrames` | `vector_eci, time` | none | Classroom-level ECI → ECEF rotation using GMST. |
| 36 | `ecef_to_eci` | `OrbitalFrames` | `vector_ecef, time` | none | Classroom-level ECEF → ECI rotation using GMST. |
| 37 | `teme_to_ecef` | `OrbitalFrames` | `r_teme_km, time` | none | Approximate TEME → ECEF position conversion for TLE/Cesium visualization. |
| 38 | `gmst_rad` | `OrbitalFrames` | `time` | none | Compute Greenwich mean sidereal time in radians. |
| 39 | `julian_date` | `OrbitalFrames` | `time` | none | Convert UTC `datetime` to Julian Date. |
| 40 | `wrap_2pi` | `OrbitalElements` | `angle_rad` | none | Wrap an angle into `[0, 2π)`. |


# Internal helper functions

These are part of the implementation, but normal coursework scripts generally should **not** call them directly.

They are included because this notebook is intended to list every function in the package.

| Function | Module | Required arguments | Description |
|---|---|---|---|
| `_vec3(value, name)` | `OrbitalElements` | both | Convert an input to a three-element numeric vector and validate size. |
| `_angle_to_rad(value, unit)` | `OrbitalElements` | both | Convert a degree/radian input to radians. |
| `_parse_element_name(name)` | `OrbitalElements` | `name` | Normalize aliases such as `theta`, `M_deg`, `h_km2_s`, etc. into canonical element names. |
| `_vec3(value, name)` | `OrbitalFrames` | both | Internal three-vector validator used by frame transforms. |
| `_as_utc(value)` | `OrbitalFrames` | `datetime` | Normalize a datetime to UTC. |
| `_state6(initial_state)` | `Propagators` | `initial_state` | Convert `StateVector` or six-value input into the six-state NumPy vector used by propagators. |
| `_stumpff_c(z)` | `Propagators` | `z` | Evaluate universal-variable Stumpff function \(C(z)\), using series expansion near zero. |
| `_stumpff_s(z)` | `Propagators` | `z` | Evaluate universal-variable Stumpff function \(S(z)\), using series expansion near zero. |
| `_initial_universal_anomaly_guess(r0_km, v0_km_s, delta_t_s, mu_km3_s2, alpha)` | `Propagators` | all five | Produce an initial Newton iteration estimate for the universal anomaly. |
| `_fixed_time_grid(t_span_s, time_step_s)` | `Propagators` | both | Build the fixed-spacing inclusive output grid used by Universal Anomaly propagation. |
| `_require_sgp4()` | `TLETools` | none | Raise a clear `ImportError` if `sgp4` is unavailable. |
| `_utc(time)` | `TLETools` | `datetime` | Normalize TLE propagation time to UTC. |
| `_utc(time)` | `CesiumOrbit` | `datetime` | Normalize Cesium trajectory time to UTC. |
| `_iso(time)` | `CesiumOrbit` | `datetime` | Convert a UTC datetime to an ISO-8601 `Z` string for CZML. |
| `_rgba(color)` | `CesiumOrbit` | RGB or RGBA iterable | Normalize a color to four integer `[R,G,B,A]` values in `[0,255]`. |



# Constants module

`Constants.py` currently contains no classes or functions, but these values are used throughout the toolkit.

| Constant | Value / units | Main use |
|---|---|---|
| `MU_EARTH_KM3_S2` | `398600.4418 km³/s²` | Earth gravitational parameter. |
| `R_EARTH_KM` | `6378.1363 km` | Earth equatorial/reference radius. |
| `J2_EARTH` | `1.08262668e-3` | Earth's J2 coefficient. |
| `OMEGA_EARTH_RAD_S` | `7.2921150e-5 rad/s` | Earth rotation rate. |
| `SECONDS_PER_DAY` | `86400 s` | Time conversion. |

The current architecture keeps these values in one module so additional sections or celestial bodies can be added without changing the function interfaces that already consume the Earth constants.



# Complete class-member inventory

This section is a compact checklist of every method/property on every class, including internal methods.

## `ClassicalOrbitalElements`

```text
__init__
_sequence_to_mapping
_prepare_input
_get_angle
_canonicalize
from_degrees

mu_km3_s2
inclination_deg
raan_deg
argument_of_periapsis_deg
true_anomaly_deg
eccentric_anomaly_rad
eccentric_anomaly_deg
mean_anomaly_rad
mean_anomaly_deg
p_km
h_km2_s
periapsis_radius_km
apoapsis_radius_km
radius_km
mean_motion_rad_s
mean_motion_rev_day
period_s
period_min
specific_orbital_energy_km2_s2

a
h
p
rp
ra
i
raan
argp
nu
theta
E
M
eccentric_anomaly
mean_anomaly

degrees
derived
vector6
to_state
```

## `StateVector`

```text
__post_init__
vector6
from_vector6
```

## `PropagationResult`

```text
positions_km
velocities_km_s
final_state
```

## `TLERecord`

```text
__post_init__
from_text
satrec
norad_id
epoch
state_at
state_at_epoch
coes_at
coes_at_epoch
propagate
```

## `CesiumScene`

```text
add_trajectory
czml
save_czml
save_html
```

## `SpaceTrackTools`

```text
__init__
status
fetch_gp_history
fetch_satcat
build_real_dataset
```



# Complete module-function inventory

This is the final completeness check.

## `OrbitalElements.py`

```text
wrap_2pi
_vec3
_angle_to_rad
_parse_element_name
eccentric_anomaly_from_true
true_anomaly_from_eccentric
mean_anomaly_from_eccentric
eccentric_anomaly_from_mean
coe_to_rv
rv_to_coe
mean_motion_rad_s
orbital_period_s
```

## `OrbitalFrames.py`

```text
_vec3
unit
rot1
rot2
rot3
dcm_from_axes
eci_to_lvlh_dcm
lvlh_to_eci_dcm
eci_to_lvlh
lvlh_to_eci
relative_state_eci_to_lvlh
perifocal_to_eci_dcm
eci_to_perifocal_dcm
eci_to_orbital_plane
orbital_plane_to_eci
_as_utc
julian_date
gmst_rad
eci_to_ecef
ecef_to_eci
teme_to_ecef
```

## `Propagators.py`

```text
two_body_acceleration
j2_acceleration
two_body_rhs
j2_rhs
_state6
_stumpff_c
_stumpff_s
_initial_universal_anomaly_guess
universal_anomaly_state
_fixed_time_grid
propagate_universal_anomaly
propagate_custom
propagate_two_body
propagate_j2
```

## `TLETools.py`

```text
_require_sgp4
_utc
parse_tle_file
```

## `CesiumOrbit.py`

```text
_utc
_iso
_rgba
trajectory_packet
times_from_seconds
```

## `SpaceTrackTools.py`

No module-level functions; functionality is exposed through the `SpaceTrackTools` class.

## `Constants.py`

No functions.

## `QuickStart.py`

No reusable functions/classes; it is an example script.

## `__init__.py`

No functions/classes are defined here; it re-exports the package's common public API.


# Suggested order to learn the toolkit

For normal Intermediate Orbits coursework, a useful progression is:

1. `ClassicalOrbitalElements`
2. `StateVector`
3. `coe_to_rv` / `rv_to_coe`
4. `propagate_two_body` with the default `DOP853` method
5. `propagate_two_body(..., method="universal_anomaly", time_step_s=...)`
6. `PropagationResult`
7. `eci_to_lvlh` and `relative_state_eci_to_lvlh`
8. `eci_to_lvlh_dcm`
9. `propagate_j2`
10. PQW/perifocal DCM functions
11. anomaly conversion helpers
12. `universal_anomaly_state` when a single future/past two-body state is needed
13. `TLERecord`
14. `CesiumScene`
15. `SpaceTrackTools`
16. custom RHS / acceleration functions
17. Earth-rotation and low-level internal helpers

That order should cover most homework and class problems before you need the less-common utilities.
